# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent   
Day 4: AutonomousPlannerAgent  
Day 5: The Price Is Right Finale


Now it's time for the Planning Agent


In [1]:
import json
from openai import OpenAI
from dotenv import load_dotenv
from agents.scanner_agent import ScannerAgent
import chromadb
import logging
load_dotenv(override=True)
openai = OpenAI()
MODEL = "gpt-5.1"


In [2]:
test_results = ScannerAgent().test_scan()
test_results

DealSelection(deals=[Deal(product_description="The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku's operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient.", price=178.0, url='https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142'), Deal(product_description='The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and

## Now let's create 3 pretend functions..

In [3]:
def scan_internet_for_deals()-> str:
    """ this tool will scan the internet for deals and return a list of deals """
    print("fake scanning the internet for deals")
    return test_results.model_dump_json()

In [4]:
def estimate_price_of_item(description: str)-> float:
    """ this tool will estimate the price of an item based on the description """
    print(f"""fake estimating the price of an item {description} always return 100.0""")
    return 100.0

In [5]:
def push_notification(description: str,deal_price: float,estimated_price: float,url: str)-> str:
    """ this tool will push a notification to the user """
    print(f"""fake pushing a notification to the user {description} always return "notification pushed" """)
    return "notification sent ok"

In [6]:
scan_internet_for_deals()

fake scanning the internet for deals


'{"deals":[{"product_description":"The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku\'s operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient.","price":178.0,"url":"https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142"},{"product_description":"The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and zoom contro

In [7]:
estimate_price_of_item("a new iphone")

fake estimating the price of an item a new iphone always return 100.0


100.0

In [8]:
push_notification("the price is right",100.0,100.0,"https://www.google.com")

fake pushing a notification to the user the price is right always return "notification pushed" 


'notification sent ok'

In [9]:
scan_function={
    "name": "scan_internet_for_deals",
    "description": "Returns top bargains scraped from the internet along with the price each item is being offered for",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}
estimate_function={
    "name": "estimate_price_of_item",
    "description": "Given the description of an item, estimate how much it is actually worth",
    "parameters": {
        "type": "object",
        "properties": {
            "description": 
            {
                "type": "string",
                "description": "A description of the item to be estimated"
            }
        },
        "required": ["description"],
        "additionalProperties": False
    }
}
push_notification_function={
    "name": "push_notification",
    "description": "Send the user a push notification about the single most compelling deal; only call this one time",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description":"description of itself scrapped from the internet"
            },
            "deal_price": {
                "type":"number",
                "description":"the price offered in deal scraped from the internet"
            },
            "estimated_price":{
                "type":"number",
                "description":"the esitmated price of the item how much it is actually worth"
            },
            "url":{
                "type":"string",
                "description":"the url of the deal scraped from the internet"
            }    
        },
        "required": ["description","deal_price","estimated_price","url"],
        "additionalProperties": False
    }
}

In [10]:
tools=[
    {"type":"function",
     "function":scan_function},
    {"type":"function",
     "function":estimate_function},
    {"type":"function",
     "function":push_notification_function}
]


In [11]:
tools

[{'type': 'function',
  'function': {'name': 'scan_internet_for_deals',
   'description': 'Returns top bargains scraped from the internet along with the price each item is being offered for',
   'parameters': {'type': 'object',
    'properties': {},
    'required': [],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'estimate_price_of_item',
   'description': 'Given the description of an item, estimate how much it is actually worth',
   'parameters': {'type': 'object',
    'properties': {'description': {'type': 'string',
      'description': 'A description of the item to be estimated'}},
    'required': ['description'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'push_notification',
   'description': 'Send the user a push notification about the single most compelling deal; only call this one time',
   'parameters': {'type': 'object',
    'properties': {'description': {'type': 'string',
      'description': 'desc

In [12]:
def hadle_tools(message):
    """
    Actually call the tools associated with this message
    """
    results=[]
    for tool_call in message.tool_calls:
        tool_name=tool_call.function.name
        arguments=json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result=tool(**arguments) if tool else {}
        results.append({"role":"tool","content":json.dumps(result),"tool_call_id":tool_call.id})
    return results

In [13]:
system_message = "You find great deals on bargain products using your tools, and notify the user of the best bargain."
user_message = """
First, use your tool to scan the internet for bargain deals. Then for each deal, use your tool to estimate its true value.
Then pick the single most compelling deal where the price is much lower than the estimated true value, and use your tool to notify the user.
Then just reply OK to indicate success.
"""
messages = [{"role": "system", "content": system_message},{"role": "user", "content": user_message}]

In [14]:
messages

[{'role': 'system',
  'content': 'You find great deals on bargain products using your tools, and notify the user of the best bargain.'},
 {'role': 'user',
  'content': '\nFirst, use your tool to scan the internet for bargain deals. Then for each deal, use your tool to estimate its true value.\nThen pick the single most compelling deal where the price is much lower than the estimated true value, and use your tool to notify the user.\nThen just reply OK to indicate success.\n'}]

In [15]:
isDone=False
while not isDone:
    response=openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
    )
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        results=hadle_tools(message)
        messages.append(message)
        messages.extend(results)
    else:
        isDone=True
print(response.choices[0].message.content)
        
        

fake scanning the internet for deals
fake estimating the price of an item The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku's operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient. always return 100.0
fake estimating the price of an item The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and zoom control, along with an ambient light sensor to adjust the vanity lighting as need

## And now.. into an Autonomous Planning Agent

And switching the fake functions for REAL functions!

In [2]:
root = logging.getLogger()
root.setLevel(logging.INFO)


In [3]:
DB ="products_vectorstore"
client = chromadb.PersistentClient(path=DB)
collection = client.get_or_create_collection("products")






In [4]:
from agents.autonomous_planning_agent import AutonomousPlanningAgent
agent = AutonomousPlanningAgent(collection)

INFO:root:[Autonomous Planning Agent] Autonomous Planning Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing
INFO:root:[Specialist Agent] Specialist Agent is ready
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using mps
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Ensemble Agent] Ensemble Agent is ready
INFO:root:[Messaging Agen

In [5]:
agent.plan()

INFO:root:[Autonomous Planning Agent] Autonomous Planning Agent is kicking off a run
INFO:root:[Autonomous Planning Agent] Autonomous Planning agent is calling scanner
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 20 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI
INFO:root:[Autonomous Planning Agent] Autonomous Planning agent is estimating value via Ensemble Agent
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
13:10:16 - LiteLLM:INFO: utils.py:3427 - 
LiteLLM completion() model= qwen2.5-coder:1.5b; provider = ollama
INFO:LiteLLM:
LiteLLM completion() model= qwen2.5-coder:1.5b; provider = ollama
13:10:19 - LiteLLM:INFO: utils.py:1307 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling succ

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $219.99
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $62.18
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $193.21
INFO:root:[Autonomous Planning Agent] Autonomous Planning agent is estimating value via Ensemble Agent
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
13:10:49 - LiteLLM:INFO: utils.py:3427 - 
LiteLLM completion() model= qwen2.5-coder:1.5b; provider = ollama
INFO:LiteLLM:
LiteLLM completion() model= qwen2.5-coder:1.5b; provider = ollama
13:10:51 - LiteLLM:INFO: utils.py:1307 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
I

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $16.99
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $114.38
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $26.93
INFO:root:[Autonomous Planning Agent] Autonomous Planning agent is estimating value via Ensemble Agent
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
13:10:53 - LiteLLM:INFO: utils.py:3427 - 
LiteLLM completion() model= qwen2.5-coder:1.5b; provider = ollama
INFO:LiteLLM:
LiteLLM completion() model= qwen2.5-coder:1.5b; provider = ollama
13:10:55 - LiteLLM:INFO: utils.py:1307 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
IN

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $69.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $91.68
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $73.27
INFO:root:[Autonomous Planning Agent] Autonomous Planning agent is estimating value via Ensemble Agent
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
13:11:01 - LiteLLM:INFO: utils.py:3427 - 
LiteLLM completion() model= qwen2.5-coder:1.5b; provider = ollama
INFO:LiteLLM:
LiteLLM completion() model= qwen2.5-coder:1.5b; provider = ollama
13:11:03 - LiteLLM:INFO: utils.py:1307 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INF

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $69.99
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $93.41
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $71.33
INFO:root:[Autonomous Planning Agent] Autonomous Planning agent is notifying user
INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
INFO:root:[Autonomous Planning Agent] Autonomous Planning Agent completed with: OK


Opportunity(deal=Deal(product_description='AZDOME M580 is a three-channel dash camera system that records in 4K for the front-facing camera and 1080p for two additional channels, providing wide coverage for vehicles. It includes a 4-inch touchscreen for live viewing and menu navigation, supports continuous loop recording, and likely offers features such as parking mode and GPS logging typical of multi-channel dash cams. The compact design mounts to the windshield and is intended for comprehensive in-car video capture for safety and evidence.', price=50.0, url='https://slickdeals.net/f/19255588-azdome-m580-3-ch-4k-1080p-1080p-dash-cam-w-4-touchscreen-50-free-s-h?utm_source=rss&utm_content=fp&utm_medium=RSS2'), estimate=193.20951632690432, discount=143.20951632690432)